[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Reading Results


## What you will be able to do

Read what a statement hands back in the shape the job needs: every row, the first, exactly one, a
single value, one column, or dictionaries ready for JSON. Reach the columns of a `Row` by position,
by name or by key, read a large result in batches, and recognize the errors from a result with no
row, with too many, or with nothing left to read.


## The idea

### The problem

The registrar's office asks the college's database different kinds of question all day. Which
student has the email `cmartin@college.edu`? Which courses has Chloe Martin taken, and what did she
get in each? How many credits has she earned? And, for the advising web page, the same transcript as
JSON. At the end of the week, all 228 enrollments go to the provost in one export. Every one of these
questions comes back from `execute` as the same kind of object, and every answer needs a different
shape: one student, a list of rows, one number, a list of dictionaries.

With `sqlite3`, every answer is a list of tuples, and the code around it fills up with index numbers
and checks: `rows[0][1]` for a name, `if len(rows) != 1` for a lookup that should find one student,
and a loop over `cursor.description` to build dictionaries for the web page. Those checks are the
ones that get left out. A lookup by name that takes `rows[0]` returns one of the two students whose
names match, without a word about the other, and `row[2]` goes on returning a column after somebody
adds another column in front of it.

### What a result is

> A **`Result`** is what `conn.execute()` returns for a `SELECT`: the rows the database sends back,
> read one at a time, and once. Each is a **`Row`**, a tuple whose values can also be reached by
> column name, as `row.name`, or through **`row._mapping`**, a dictionary that cannot be changed.
> The result's methods choose the shape of the answer. **`all()`** returns every row in a list,
> **`first()`** the first row or `None`, **`one()`** the only row, raising an error if there are
> none or several, and **`one_or_none()`** the only row or `None`. **`scalar_one()`** returns the
> first column of the only row, **`scalars()`** the first column of every row, **`mappings()`**
> every row as a dictionary, and **`partitions()`** the rows in batches.

### Why it works that way

- **A result is a stream, not a list.** Rows come from the database as they are read, which lets a
  report of a million rows run in the memory it takes to hold a hundred. The price is that a result
  can be read only once: after `all()` there is nothing left to read, and `first()` and `one()` close
  the result behind them.
- **A row is a tuple first.** It unpacks, indexes and compares like one, so
  `for code, title, credits in result` works. Its names come from the columns of the query, and
  `_mapping`, `_fields` and `_asdict()` start with an underscore so that no column's name can clash
  with them.
- **`one()` states what the program expects.** A lookup by a unique email must find exactly one
  student, and `one()` raises when it finds none or several. `first()` takes whatever came first and
  says nothing about the rest, and so does `scalar()`.
- **The shape is chosen at the end.** `scalars()`, `mappings()` and `columns()` return new results
  over the same rows, so the statement stays the same and only the last call changes.
- **The ORM hands back the same `Result`.** `session.execute()` returns this kind of object, and
  `session.scalars()` is `execute()` followed by `scalars()`, so all of this carries over to the
  notebooks about the ORM.

### Where this shows up

The **Declarative Models** notebook runs `select(Student)` through a session, and the `Result` it
gets back holds `Student` objects in its rows, which is why `session.scalars()` is the ORM's usual
way of reading. A web service that answers with JSON builds the answer from `mappings()`, one
dictionary for every row. Pandas builds a DataFrame from `result.all()` and `result.keys()`, or
reads a query straight into one with `read_sql`. The **Row Factories** notebook of the
**sqlite3, Deep Dive** guide did the same job one level down, with `sqlite3.Row` and factories of
its own.

### What this notebook covers

- A `Result`, its columns and its rows
- Three ways into a `Row`: position, name and `_mapping`
- `all()`, `first()`, `one()` and `one_or_none()`
- One value and one column: `scalar_one()`, `scalar()` and `scalars()`
- Dictionaries for JSON, with `mappings()`
- A large result in batches, with `partitions()`
- Which method answers which question
- A transcript for the advising page, finished
- Six errors, from a lookup that found nobody to dictionaries that JSON cannot write

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
from sqlalchemy import create_engine, text

engine = create_engine("sqlite://")
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE courses (code TEXT, title TEXT, credits INTEGER)"))
    conn.execute(text("INSERT INTO courses VALUES ('BIO-101', 'Introduction to Biology', 4), "
                      "('STA-200', 'Statistics', 3)"))

    row = conn.execute(text("SELECT code, title, credits FROM courses ORDER BY code")).first()
    print(row.code, "|", row[1], "|", row._mapping["credits"])
    print(conn.execute(text("SELECT title FROM courses ORDER BY code")).scalars().all())
    print(conn.execute(text("SELECT SUM(credits) FROM courses")).scalar_one())
```

```
BIO-101 | Introduction to Biology | 4
['Introduction to Biology', 'Statistics']
7
```

One row, reached by name, by position and by key; one column of every row as a list; and one value.
The statements are ordinary SQL, and the method after `execute` chose the shape of each answer.


## Setup

Seven imports, the college's database, and the engine helper.

- `sqlalchemy` is the library itself, and the cell prints its version
- `create_engine`, `event` and `text`, from `sqlalchemy`, make the engine, switch on its foreign
  keys, and run SQL
- `StaticPool`, from `sqlalchemy.pool`, is the pool the helper uses for a database in memory
- `json` writes a transcript the way a web page would receive it
- `sqlite3` builds the database, `Path` names the files, and `shutil` removes the scratch folder at
  the start and at the end

The database, `scratch/college.db`, is the one the **Connections and Transactions** notebook built:
25 students, 10 courses, 4 terms, a section of every course in every term, and 228 enrollments,
completed with a grade in the first three terms and under way, with no grade yet, in Spring 2026.
`college_engine` is that notebook's engine helper.

Colab has SQLAlchemy installed, and this notebook runs version 2.0.54. Any 2.0 release runs it,
though an error may be worded a little differently. To match it exactly, run
`%pip install sqlalchemy==2.0.54` in a cell of its own, restart the session, and run this cell again.


In [1]:
import json
import shutil
import sqlite3
from pathlib import Path

import sqlalchemy
from sqlalchemy import create_engine, event, text
from sqlalchemy.pool import StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
TERMS = [("Fall 2024", "2024-08-26"), ("Spring 2025", "2025-01-13"), ("Fall 2025", "2025-08-25"),
         ("Spring 2026", "2026-01-12")]
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], TERMS[i % 3][1]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]
GRADES = ["A", "A-", "B+", "B", "B-", "C+", "C", "C-", "D", "F"]

# One section of every course in every term, so the section of course c in term t has id (t - 1) * 10 + c.
SECTIONS = [(course, term, 30) for term in range(1, len(TERMS) + 1) for course in range(1, len(COURSES) + 1)]

# Three courses a term for every student, from the term they started. Spring 2026 is under way.
ENROLLMENTS = []
for s in range(len(NAMES)):
    for term in range(s % 3 + 1, len(TERMS) + 1):
        for k in range(3):
            section = (term - 1) * len(COURSES) + (s + term + 3 * k) % len(COURSES) + 1
            if term < len(TERMS):
                ENROLLMENTS.append((s + 1, section, "completed", GRADES[(s * 7 + term * 5 + k * 3) % len(GRADES)]))
            else:
                ENROLLMENTS.append((s + 1, section, "enrolled", None))

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
    CREATE TABLE terms (id INTEGER PRIMARY KEY, name TEXT NOT NULL UNIQUE, starts_on TEXT NOT NULL);
    CREATE TABLE sections (id INTEGER PRIMARY KEY, course_id INTEGER NOT NULL REFERENCES courses (id),
                           term_id INTEGER NOT NULL REFERENCES terms (id), capacity INTEGER NOT NULL);
    CREATE TABLE enrollments (student_id INTEGER NOT NULL REFERENCES students (id),
                              section_id INTEGER NOT NULL REFERENCES sections (id),
                              status TEXT NOT NULL, grade TEXT,
                              PRIMARY KEY (student_id, section_id));
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.executemany("INSERT INTO terms (name, starts_on) VALUES (?, ?)", TERMS)
build.executemany("INSERT INTO sections (course_id, term_id, capacity) VALUES (?, ?, ?)", SECTIONS)
build.executemany("INSERT INTO enrollments (student_id, section_id, status, grade) VALUES (?, ?, ?, ?)", ENROLLMENTS)
build.commit()
build.close()

def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, echo=echo,
                               connect_args={"check_same_thread": False, "autocommit": False})
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo, connect_args={"autocommit": False})

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.autocommit = True          # the PRAGMA does nothing inside a transaction,
        dbapi_connection.execute("PRAGMA foreign_keys = ON")
        dbapi_connection.autocommit = False         # and with autocommit=False sqlite3 keeps one open

    return engine


engine = college_engine(DATABASE)
print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "|", len(STUDENTS), "students,", len(ENROLLMENTS), "enrollments")


sqlalchemy 2.0.54 | scratch/college.db | 25 students, 228 enrollments


## Worked examples

### A Result, and the rows in it

The queries this notebook runs, written once with `text()`. `TRANSCRIPT` lists a student's courses,
term by term, with the grade of every course finished. Run it for Chloe Martin, student 3, and loop
over what `execute` returns:


In [2]:
TRANSCRIPT = text("""
    SELECT terms.name AS term, courses.code, courses.title, courses.credits, enrollments.grade
    FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    JOIN terms ON terms.id = sections.term_id
    WHERE enrollments.student_id = :student
    ORDER BY terms.starts_on, courses.code
""")
BY_EMAIL = text("SELECT id, name, program FROM students WHERE email = :email")
BY_PROGRAM = text("SELECT id, name FROM students WHERE program = :program ORDER BY name")
CREDITS_EARNED = text("""
    SELECT SUM(courses.credits) FROM enrollments
    JOIN sections ON sections.id = enrollments.section_id
    JOIN courses ON courses.id = sections.course_id
    WHERE enrollments.student_id = :student AND enrollments.status = 'completed' AND enrollments.grade <> 'F'
""")

with engine.connect() as conn:
    result = conn.execute(TRANSCRIPT, {"student": 3})
    print(type(result).__name__, "with the columns", list(result.keys()))
    for row in result:
        print(row)


CursorResult with the columns ['term', 'code', 'title', 'credits', 'grade']
('Fall 2025', 'CHE-110', 'General Chemistry', 4, 'C+')
('Fall 2025', 'CSC-201', 'Data Structures', 3, 'F')
('Fall 2025', 'PSY-101', 'Introduction to Psychology', 3, 'B+')
('Spring 2026', 'ENG-105', 'Composition', 3, None)
('Spring 2026', 'MAT-120', 'Calculus I', 4, None)
('Spring 2026', 'STA-200', 'Statistics', 3, None)


`execute` returned a `CursorResult`, SQLAlchemy's `Result` for a statement that went to the database,
and `keys()` names its columns: the names in the `SELECT`, with `terms.name` renamed `term` by `AS`.
Every row prints as a tuple. Chloe started in Fall 2025, so her transcript has two terms, and the
three Spring 2026 courses have `None` for a grade, which is how SQL's `NULL` arrives in Python.

### Three ways into a Row


In [3]:
with engine.connect() as conn:
    row = conn.execute(TRANSCRIPT, {"student": 3}).first()

print("by position:", row[1], row[4])
print("by name:    ", row.code, row.grade)
print("by key:     ", row._mapping["code"], row._mapping["grade"])

term, code, title, credits, grade = row
print("unpacked:   ", term, code, credits)
print("its columns:", row._fields)
print("as a dict:  ", row._asdict())


by position: CHE-110 C+
by name:     CHE-110 C+
by key:      CHE-110 C+
unpacked:    Fall 2025 CHE-110 4
its columns: ('term', 'code', 'title', 'credits', 'grade')
as a dict:   {'term': 'Fall 2025', 'code': 'CHE-110', 'title': 'General Chemistry', 'credits': 4, 'grade': 'C+'}


Position is the tuple's way, and depends on the order of the columns in the `SELECT`. A name reads
better and survives a column added in front of it. `row._mapping` is a dictionary view of the row,
which suits code that is handed a column's name as a string, and `_asdict()` copies the row into an
ordinary dictionary. Unpacking works because a row is a tuple, so the number of names has to match
the number of columns. The row is still usable after the block ended: its values were read out of
the database when `first()` fetched it.

### all(), first(), one() and one_or_none()

How many rows the program expects decides the method. A search by program expects several, a lookup
by email exactly one, and a lookup that may miss, one or none:


In [4]:
with engine.connect() as conn:
    print("all():        ", conn.execute(BY_PROGRAM, {"program": "History"}).all())
    print("first():      ", conn.execute(BY_PROGRAM, {"program": "History"}).first())
    print("one():        ", conn.execute(BY_EMAIL, {"email": "cmartin@college.edu"}).one())
    print("one_or_none():", conn.execute(BY_EMAIL, {"email": "nobody@college.edu"}).one_or_none())


all():         [(25, "Aoife O'Brien"), (5, 'Elena Petrova'), (10, 'Jonas Berg'), (15, 'Olivia Brandt'), (20, 'Tara Nilsen')]
first():       (25, "Aoife O'Brien")
one():         (3, 'Chloe Martin', 'Mathematics')
one_or_none(): None


`all()` returned the five History students in a list. `first()` returned the first of them and
discarded the other four. `one()` returned Chloe Martin's row, and would have raised an error had the
email matched no student or two, which Common errors shows. `one_or_none()` returned `None` for an
email nobody has, which leaves the program to decide what a missing student means.

### One value, and one column

A count, a sum or an id is a single value: the first column of a single row. `scalar_one()` returns
it, and `scalars()` returns one column of every row, the first unless it is given another:


In [5]:
with engine.connect() as conn:
    print("credits Chloe earned:  ", conn.execute(CREDITS_EARNED, {"student": 3}).scalar_one())
    print("History students:      ", conn.execute(BY_PROGRAM, {"program": "History"}).scalars("name").all())
    print("an email nobody has:   ", conn.execute(BY_EMAIL, {"email": "nobody@college.edu"}).scalar_one_or_none())
    print("scalar() on five rows: ", conn.execute(BY_PROGRAM, {"program": "History"}).scalar())


credits Chloe earned:   7
History students:       ["Aoife O'Brien", 'Elena Petrova', 'Jonas Berg', 'Olivia Brandt', 'Tara Nilsen']
an email nobody has:    None
scalar() on five rows:  25


Chloe finished three courses in Fall 2025, of 4, 3 and 3 credits, and failed the Data Structures
course worth 3, so she earned 7. `scalars("name")` took the `name` column of every row, and
`scalar_one_or_none()` is the one-or-none form for a single value. The last line is the one to
watch: `scalar()` took the first column of the first row, 25, which is Aoife O'Brien's id, and
discarded the other four rows without a word. `scalar_one()` would have raised, so it is the safer
choice whenever a query is meant to return exactly one row.

### Dictionaries for JSON

`mappings()` returns every row as a `RowMapping`, a dictionary that cannot be changed. A web page
wants JSON, and `json.dumps` writes plain dictionaries, so `dict()` makes one of each:


In [6]:
MATH_COURSES = text("SELECT code, title, credits FROM courses WHERE department = 'Mathematics' ORDER BY code")

with engine.connect() as conn:
    math = conn.execute(MATH_COURSES).mappings().all()

print(type(math[0]).__name__, "|", math[0]["title"], "|", list(math[0].keys()))
print(json.dumps([dict(course) for course in math], indent=2))


RowMapping | Calculus I | ['code', 'title', 'credits']
[
  {
    "code": "MAT-120",
    "title": "Calculus I",
    "credits": 4
  },
  {
    "code": "MAT-121",
    "title": "Calculus II",
    "credits": 4
  },
  {
    "code": "STA-200",
    "title": "Statistics",
    "credits": 3
  }
]


A `RowMapping` answers to `["title"]`, `keys()` and the other ways a dictionary is read, and
`dict(course)` copies it into one that `json.dumps` can write. The last of the Common errors shows
what `json.dumps` does with a `RowMapping` itself.

### A large result in batches

`all()` holds every row in memory at once. `partitions()` hands them over a batch at a time, which is
how an export of every enrollment, or of millions, runs without holding them all:


In [7]:
EVERY_ENROLLMENT = text("SELECT student_id, section_id, status, grade FROM enrollments ORDER BY student_id, section_id")

with engine.connect() as conn:
    result = conn.execute(EVERY_ENROLLMENT)
    for number, batch in enumerate(result.partitions(100), start=1):
        print(f"batch {number}: {len(batch):>3} rows, from {batch[0]} to {batch[-1]}")


batch 1: 100 rows, from (1, 2, 'completed', 'C+') to (11, 31, 'enrolled', None)
batch 2: 100 rows, from (11, 35, 'enrolled', None) to (22, 36, 'enrolled', None)
batch 3:  28 rows, from (22, 39, 'enrolled', None) to (25, 39, 'enrolled', None)


Three batches, 100, 100 and 28 rows, and at no point more than a hundred in the program's hands.
`partitions()` reads them with the driver's `fetchmany()`, which the result offers too, and the loop
body would write each batch to a file before asking for the next.

### Which method answers which question

| Use | When the question is | What comes back |
|---|---|---|
| `all()` | every row, such as a transcript or a search | a list of `Row` |
| `one()` | the one row a unique key must find, such as a student by email | a `Row`, or an error for none or several |
| `one_or_none()` | one row that may not be there | a `Row` or `None`, or an error for several |
| `first()` | any one row, when the rest do not matter | the first `Row` or `None`, the rest discarded |
| `scalar_one()` | one value, such as a count, a sum or an id | the value, or an error for none or several rows |
| `scalars()` | one column, such as the codes of some courses | a result of values, read with `all()`, `first()` or `one()` |
| `mappings()` | rows as dictionaries, for JSON or for code handed column names | a result of `RowMapping` |
| `partitions(n)` | more rows than fit comfortably in memory | lists of up to `n` rows |

The defaults are `all()` for a list, `one()` for a lookup by a unique key, and `scalar_one()` for a
single value. `first()` and `scalar()` are for the case where more rows would not be a mistake.

### A transcript for the advising page, finished

The pieces of this notebook in one function. `transcript` looks a student up by email with
`one_or_none()`, reads their courses as dictionaries with `mappings()`, asks the database for the
credits they earned with `scalar_one()`, works out a grade point average from the courses with a
grade, and returns a dictionary ready for JSON, or `None` for an email nobody has:


In [8]:
GRADE_POINTS = {"A": 4.0, "A-": 3.7, "B+": 3.3, "B": 3.0, "B-": 2.7, "C+": 2.3, "C": 2.0, "C-": 1.7, "D": 1.0, "F": 0.0}


def transcript(conn, email):
    """A student's transcript as a dictionary ready for JSON, or None for an email nobody has."""
    student = conn.execute(BY_EMAIL, {"email": email}).mappings().one_or_none()
    if student is None:
        return None
    courses = conn.execute(TRANSCRIPT, {"student": student["id"]}).mappings().all()
    graded = [course for course in courses if course["grade"] is not None]
    attempted = sum(course["credits"] for course in graded)
    points = sum(GRADE_POINTS[course["grade"]] * course["credits"] for course in graded)
    return {
        "name": student["name"],
        "program": student["program"],
        "credits_attempted": attempted,
        "credits_earned": conn.execute(CREDITS_EARNED, {"student": student["id"]}).scalar_one(),
        "gpa": round(points / attempted, 2) if attempted else None,
        "courses": [dict(course) for course in courses],
    }



with engine.connect() as conn:
    record = transcript(conn, "cmartin@college.edu")
    nobody = transcript(conn, "nobody@college.edu")

print(json.dumps({key: value for key, value in record.items() if key != "courses"}))
for course in record["courses"]:
    print("   ", json.dumps(course))
print("an email nobody has:", nobody)


{"name": "Chloe Martin", "program": "Mathematics", "credits_attempted": 10, "credits_earned": 7, "gpa": 1.91}
    {"term": "Fall 2025", "code": "CHE-110", "title": "General Chemistry", "credits": 4, "grade": "C+"}
    {"term": "Fall 2025", "code": "CSC-201", "title": "Data Structures", "credits": 3, "grade": "F"}
    {"term": "Fall 2025", "code": "PSY-101", "title": "Introduction to Psychology", "credits": 3, "grade": "B+"}
    {"term": "Spring 2026", "code": "ENG-105", "title": "Composition", "credits": 3, "grade": null}
    {"term": "Spring 2026", "code": "MAT-120", "title": "Calculus I", "credits": 4, "grade": null}
    {"term": "Spring 2026", "code": "STA-200", "title": "Statistics", "credits": 3, "grade": null}
an email nobody has: None


Chloe Martin attempted 10 credits and earned 7, and her average is 1.91: 2.3 points for each of the 4
credits of General Chemistry, 3.3 for each of the 3 of Introduction to Psychology, and nothing for
the F in Data Structures, over the 10 credits attempted. Her Spring 2026 courses have no grade yet,
so they count toward neither figure, and they reach the JSON as `null`.

### Where each part came from

| In `transcript` | What it relies on | The section that showed it |
|---|---|---|
| `mappings().one_or_none()` for the student | one row or `None`, as a dictionary | `all()`, `first()`, `one()` and `one_or_none()`; Dictionaries for JSON |
| `mappings().all()` for the courses | every row, as dictionaries | Dictionaries for JSON |
| `scalar_one()` for the credits earned | one value, the first column of the only row | One value, and one column |
| `course["grade"] is not None` | SQL's `NULL` arriving as `None` | A Result, and the rows in it |
| `[dict(course) for course in courses]` | plain dictionaries that `json.dumps` can write | Dictionaries for JSON |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/04-reading-results-solutions.ipynb).

**1.** With `one()`, print the name and program of the student whose email is `hali@college.edu`,
reaching the columns by name.


In [9]:
# your code here


**2.** With `scalars()`, list the codes of the courses worth 4 credits.


In [10]:
# your code here


**3.** With `scalar_one()`, count the enrollments in Spring 2026, whose sections are 31 to 40.


In [11]:
# your code here


**4.** Print the Computer Science courses as dictionaries with `mappings()`, then as JSON.


In [12]:
# your code here


**5.** Read every student in batches of 10 with `partitions()`, and print how many rows each batch
holds and the first name in it.


In [13]:
# your code here


**6.** With `transcript`, print the grade point average of every student who started in Fall 2025,
highest first.


In [14]:
# your code here


## Common errors

### sqlalchemy.exc.NoResultFound: No row was found when one was required


In [15]:
def student_id(conn, email):
    """The id of the student with this email."""
    return conn.execute(BY_EMAIL, {"email": email}).scalar_one()


with engine.connect() as conn:
    print(student_id(conn, "cmartin@college.edu"))
    print(student_id(conn, "c.martin@college.edu"))


3


NoResultFound: No row was found when one was required

The first email found Chloe Martin. The second, mistyped, found nobody, and `scalar_one()`, like
`one()`, raises when there is no row, because it was told to expect one. That is the right error for
a key that must exist. When a missing row is an answer rather than a mistake, as it is for an email
somebody typed into a form, ask for one or none, and decide what `None` means:


In [16]:
def student_id(conn, email):
    """The id of the student with this email, or None if nobody has it."""
    return conn.execute(BY_EMAIL, {"email": email}).scalar_one_or_none()


with engine.connect() as conn:
    print(student_id(conn, "cmartin@college.edu"), student_id(conn, "c.martin@college.edu"))


3 None


### sqlalchemy.exc.MultipleResultsFound: Multiple rows were found when exactly one was required


In [17]:
STARTS_WITH = text("SELECT id, name FROM students WHERE name LIKE :start || '%' ORDER BY name")

with engine.connect() as conn:
    print(conn.execute(STARTS_WITH, {"start": "Ch"}).one())
    print(conn.execute(STARTS_WITH, {"start": "A"}).one())


(3, 'Chloe Martin')


MultipleResultsFound: Multiple rows were found when exactly one was required

`Ch` matched one student and `A` matched two, Ana Reyes and Aoife O'Brien, and `one()` refused to
pick one of them. `first()` would have picked Ana, the first by name, without mentioning Aoife, which
is the mistake `one()` exists to catch. A name is not a key: return every match and let whoever asked
choose, or look the student up by something unique, such as the email:


In [18]:
with engine.connect() as conn:
    print(conn.execute(STARTS_WITH, {"start": "A"}).all())


[(1, 'Ana Reyes'), (25, "Aoife O'Brien")]


### AttributeError: Could not locate column in row for column 'keys'


In [19]:
with engine.connect() as conn:
    row = conn.execute(BY_EMAIL, {"email": "cmartin@college.edu"}).one()

print({key: row[key] for key in row.keys()})


AttributeError: Could not locate column in row for column 'keys'

A `Row` is a tuple, not a dictionary, and it reads any attribute it does not have as the name of a
column, so `row.keys` went looking for a column called `keys`. The **Row Factories** notebook of the
**sqlite3, Deep Dive** guide used `sqlite3.Row`, which does have `keys()`, and that is where the
habit comes from. The dictionary side of a row is `_mapping`:


In [20]:
print({key: row._mapping[key] for key in row._mapping.keys()})
print(dict(row._mapping) == row._asdict())


{'id': 3, 'name': 'Chloe Martin', 'program': 'Mathematics'}
True


### No error, and an empty list: a result read twice


In [21]:
with engine.connect() as conn:
    result = conn.execute(BY_PROGRAM, {"program": "History"})
    print(len(result.all()), "History students")
    names = [row.name for row in result.all()]

print("their names:", names)


5 History students
their names: []


Five students, and then no names. The first `all()` read every row, and the second found nothing left
to read, since a result is a stream that runs once. Nothing raised, because an empty list is a
perfectly good answer to "the rows that are left". Read the result once and keep the list:


In [22]:
with engine.connect() as conn:
    rows = conn.execute(BY_PROGRAM, {"program": "History"}).all()

print(len(rows), "History students:", [row.name for row in rows])


5 History students: ["Aoife O'Brien", 'Elena Petrova', 'Jonas Berg', 'Olivia Brandt', 'Tara Nilsen']


### sqlalchemy.exc.ResourceClosedError: This result object is closed.


In [23]:
with engine.connect() as conn:
    result = conn.execute(BY_PROGRAM, {"program": "History"})
    print("the first History student:", result.first())
    print("the others:", result.all())


the first History student: (25, "Aoife O'Brien")


ResourceClosedError: This result object is closed.

`first()` returns one row and closes the result, discarding the rows after it, so there was nothing
for `all()` to read. To have the first row and the rest, read them all once, and split the list:


In [24]:
with engine.connect() as conn:
    first, *others = conn.execute(BY_PROGRAM, {"program": "History"}).all()

print("the first History student:", first)
print("the others:", others)


the first History student: (25, "Aoife O'Brien")
the others: [(5, 'Elena Petrova'), (10, 'Jonas Berg'), (15, 'Olivia Brandt'), (20, 'Tara Nilsen')]


### TypeError: Object of type RowMapping is not JSON serializable


In [25]:
with engine.connect() as conn:
    courses = conn.execute(text("SELECT code, credits FROM courses WHERE credits = 4 ORDER BY code")).mappings().all()

json.dumps(courses)


TypeError: Object of type RowMapping is not JSON serializable

A `RowMapping` behaves like a dictionary, but it is not a `dict`, and `json.dumps` writes only the
types it knows: dictionaries, lists, strings, numbers, `True`, `False` and `None`. Copy every row
into a plain dictionary first:


In [26]:
print(json.dumps([dict(course) for course in courses]))


[{"code": "BIO-101", "credits": 4}, {"code": "CHE-110", "credits": 4}, {"code": "MAT-120", "credits": 4}, {"code": "MAT-121", "credits": 4}]


Last, the engine lets go of the file, and this cell removes the scratch folder, with the database in
it:


In [27]:
engine.dispose()
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


## Recap

- `execute` returns a `Result`, read once, as a stream, and `all()`, `first()`, `one()` and
  `one_or_none()` say how many rows the program expects.
- A `Row` is a tuple with names: reach a column by position, by `row.name`, or by key through
  `row._mapping`.
- `scalar_one()` returns one value, `scalars()` one column, and `mappings()` dictionaries, which
  `dict()` turns into what `json.dumps` can write.
- `partitions()` reads a large result in batches, so the program never holds all of it at once.
- `one()` and `scalar_one()` raise for no row and for several, where `first()` and `scalar()` take
  the first row and say nothing about the rest.


## What is next

The **Tables and Metadata** notebook describes the college's tables in Python instead of SQL text:
`Table` and `MetaData`, constraints with names, and reflecting a database somebody else designed.


---

&#8592; **Previous:** [Connections and Transactions](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/03-connections-and-transactions.ipynb)  &nbsp;·&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
